# langchain-ivoire quick start: talk to any chat model in Baoulé

`with_language(llm, lang="bci")` wraps a chat model:

- The user's Baoulé is translated into French.
- The model answers in short, simple French.
- The answer is translated back into Baoulé.

The model never sees Baoulé, so any chat model works, hosted or local.

This notebook runs **offline**:

- `FakeBackend` stands in for the translation API.
- `GenericFakeChatModel` stands in for the LLM.

The real lines are shown in comments. Swap them in once you have an API key (`IVOIRE_API_KEY`) and an LLM key.

In [1]:
from langchain_core.language_models.fake_chat_models import GenericFakeChatModel

from langchain_ivoire import FakeBackend, with_language

# The canned translations are entries of the demo phrasebook (automatic checks only).
# Real setup (needs keys):
#   from langchain.chat_models import init_chat_model
#   llm = init_chat_model("provider:model")  # any chat model
#   backend = None   # -> remote API, key from IVOIRE_API_KEY
llm = GenericFakeChatModel(messages=iter(["Je vais bien, merci.", "Venez manger."]))
backend = FakeBackend(
    translations={
        ("bci", "fra", "N tɛ, an ti sɛn?"): "Bonjour, comment allez-vous ?",
        ("fra", "bci", "Je vais bien, merci."): "N liɛ yo'a kpa, n yo.",
        ("bci", "fra", "Bla di like."): "Venez manger.",
        ("fra", "bci", "Venez manger."): "Bla di like.",
    }
)

chat = with_language(llm, lang="bci", backend=backend)
reply = chat.invoke("N tɛ, an ti sɛn?")  # "Hello, how are you?"
print(reply.content)

N liɛ yo'a kpa, n yo.


In [2]:
meta = reply.response_metadata["ivoire"]
print("French sent to the model:", meta["input_fra"])
print("French reply            :", meta["output_fra"])
print("timings (ms)            :", sorted(meta["timings_ms"]))

French sent to the model: Bonjour, comment allez-vous ?
French reply            : Je vais bien, merci.
timings (ms)            : ['llm', 'total', 'translate_in', 'translate_out']


It is a normal chat model, so message lists and history work. Earlier assistant turns are not translated again: their French is taken from the metadata.

In [3]:
from langchain_core.messages import HumanMessage

second = chat.invoke([HumanMessage("N tɛ, an ti sɛn?"), reply, HumanMessage("Bla di like.")])
print(second.content, "|", second.response_metadata["ivoire"]["output_fra"])

Bla di like. | Venez manger.


## The building blocks

The transcriber, translator and speaker are Runnables, so they compose with `|`.

In [4]:
from langchain_ivoire import IvoireSpeaker, IvoireTranslator

be = FakeBackend(translations={("fra", "bci", "Venez manger."): "Bla di like."})
say = IvoireTranslator(lang="bci", source="fra", target="bci", backend=be) | IvoireSpeaker(
    lang="bci", backend=be
)
speech = say.invoke("Venez manger.")
print(speech.text, speech.media_type, f"{speech.audio_s} s", len(speech.audio), "bytes")

Bla di like. audio/wav 0.92 s 29484 bytes


## Voice agent (LangGraph)

The voice agent runs these steps:

1. Listen (speech recognition).
2. Snap to the phrasebook when the utterance is clearly a known phrase.
3. Translate into French.
4. Ask the LLM.
5. Translate the reply back.
6. Speak the reply.

The canned translations are demo phrasebook entries. Here the fake ASR "hears" a learner's *bla de le kɛ*, which snaps to the phrasebook entry *Bla di like.* ("Come and eat.").

In [5]:
from langchain_core.language_models.fake_chat_models import FakeListChatModel

from langchain_ivoire import create_voice_agent
from langchain_ivoire._audio import silent_wav

be = FakeBackend(
    transcripts=["bla de le kɛ"],
    translations={("fra", "bci", "Je vous remercie."): "Yɛ ɛ lɛ'n wie."},
)
agent = create_voice_agent(
    FakeListChatModel(responses=["Je vous remercie."]),
    lang="bci",
    phrasebook="../tests/fixtures/phrasebook_bci.json",
    backend=be,
)
out = agent.invoke({"audio": silent_wav(1.0)})
print("route     :", " -> ".join(out["route"]))
print("heard     :", out["heard"], "=>", out["transcript"], f"(match {out['phrase']['score']})")
print("to the LLM:", out["user_fra"])
print(
    "reply     :", out["reply_fra"], "=>", out["reply"], f"+ {len(out['reply_audio'])} bytes of WAV"
)

route     : listen -> phrase_snap -> agent -> translate_out -> speak -> out
heard     : bla de le kɛ => Bla di like. (match 0.778)
to the LLM: Venez manger.
reply     : Je vous remercie. => Yɛ ɛ lɛ'n wie. + 33324 bytes of WAV
